# 33 · Country, local time, holidays, industry; where promo over-performs; brand popularity on Wikipedia (E84-E86)

Over-performance r = true − out-of-fold final model (log10). Outputs carry statistics only.

In [1]:
import sys, json, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw" / "reach_samples_students.csv").exists())
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.width", 250, "display.max_columns", 30, "display.max_rows", 300)
import extra_models as em, eda_utils as eu, history as H, account_clusters as ac, author_more as am, author_next as an, context_more as cm, context_text as ct

In [2]:
f, meta = em.load(ROOT); raw_all = eu.load(ROOT); raw = raw_all.loc[f.index]
pred = ac.oof(f, meta); d = am.frame(raw, meta, pred)
E = ct.embeddings(ROOT, ROOT / "data" / "processed" / "embeddings_e5_cache.pkl"); G, _ = ac.brand_graph(raw_all, E); lab_ind, T69, info = ac.communities(G, raw_all, None); print(info)
labels = pd.read_csv(ROOT / "data" / "processed" / "mention_labels.csv", dtype={"mention_id": str}).set_index("mention_id").type
lab = pd.Series(labels.reindex(raw.mention_id.astype(str)).values, index=raw.index); kind = ac.account_kind(raw)

{'edges': 2029, 'edges with shared authors': 539, 'communities': 9, 'modularity': 0.5027031451525458}


## E84 · Country, local time, local holidays, industry as columns

In [3]:
LC = cm.local_context(raw); IND = cm.industry(meta.brand, lab_ind)
print("posts with a local time:", (LC.local_hour >= 0).groupby(meta.platform).mean().round(2).to_dict())
TY = pd.DataFrame({f"type_{t}": (lab == t).astype(int) for t in ["promo", "spam", "unclear"]}, index=raw.index)
GR = {"+ country": LC.filter(like="country_"), "+ local time and holidays": LC[["local_hour", "local_weekend", "local_holiday"]], "+ industry": IND, "+ type label": TY,
      "+ all four": pd.concat([LC, IND, TY], axis=1)}
P84 = cm.variants(f, meta, GR); S84 = H.summarise(P84); display(S84.pivot(index="model", columns="platform", values="MAE")[cm.PLATS].round(3))
G84 = pd.DataFrame([{"platform": p, "model": k, **dict(zip(["gain", "CI low", "CI high"], H.gain_ci(P84, p, "final model", k)))} for p in cm.PLATS for k in list(GR)]); G84.round(3)

posts with a local time: {'tiktok': 0.91, 'twitter': 0.21, 'youtube': 0.78}


platform,tiktok,youtube,twitter
model,,,
+ all four,0.267,0.461,0.468
+ country,0.269,0.456,0.468
+ industry,0.274,0.461,0.471
+ local time and holidays,0.271,0.458,0.471
+ type label,0.269,0.451,0.463
final model,0.273,0.456,0.466


,platform,model,gain,CI low,CI high
0,tiktok,+ country,0.005,0.002,0.008
1,tiktok,+ local time and holidays,0.002,-0.001,0.005
2,tiktok,+ industry,-0.001,-0.002,0.001
3,tiktok,+ type label,0.004,0.002,0.007
4,tiktok,+ all four,0.007,0.003,0.010
5,youtube,+ country,-0.000,-0.004,0.003
6,youtube,+ local time and holidays,-0.002,-0.007,0.003
7,youtube,+ industry,-0.005,-0.009,-0.002
8,youtube,+ type label,0.005,0.001,0.008
9,youtube,+ all four,-0.005,-0.011,0.000


## E85 · Where exactly promo over-performs
Promo minus organic over-performance (our AI labels) within subgroups; and where adding the label to the final model helps.

In [4]:
R85 = cm.promo_subgroups(d, f, meta, raw, lab, kind, lab_ind); R85.round(3)

,platform,split,group,promo posts,organic posts,promo share,promo − organic,CI low,CI high
0,tiktok,all,all posts,844,1179,0.417,0.092,0.058,0.128
1,tiktok,tier,100k-1M,61,47,0.565,0.133,-0.000,0.286
2,tiktok,tier,10k-100k,158,217,0.421,0.177,0.069,0.272
3,tiktok,tier,1k-10k,266,412,0.392,0.085,0.026,0.152
4,tiktok,tier,< 1k,338,493,0.407,0.047,-0.004,0.094
5,tiktok,react,1-10,220,184,0.545,0.077,-0.018,0.166
6,tiktok,react,101-1000,154,281,0.354,0.137,0.068,0.213
7,tiktok,react,11-100,351,584,0.375,0.137,0.092,0.181
8,tiktok,react,>1000,69,117,0.371,0.107,-0.000,0.235
9,tiktok,kind,"brand in the name (branch, dealer, fan)",37,34,0.521,0.088,-0.120,0.326


In [5]:
a = P84[P84.model == "final model"].reset_index(drop=True); key = lab.reindex(a.idx).fillna("none").values
L85 = cm.label_gain_by(P84, key, "+ type label"); display(L85.round(4))
rx = pd.cut(__import__("reaction_tiers").reactions(f).reindex(a.idx), __import__("reaction_tiers").EDGES, labels=__import__("reaction_tiers").TIERS).astype(str).values
L85b = cm.label_gain_by(P84, np.char.add(np.char.add(key.astype(str), " · "), rx), "+ type label"); L85b[L85b.posts >= 30].round(4)

,platform,key,posts,gain
0,tiktok,organic,1179,0.0027
1,tiktok,promo,844,0.0080
2,tiktok,spam,331,0.0031
3,tiktok,unclear,137,-0.0052
4,twitter,organic,1242,0.0041
5,twitter,promo,334,0.0121
6,twitter,spam,212,-0.0077
7,twitter,unclear,15,-0.0522
8,youtube,organic,734,-0.0028
9,youtube,promo,1037,0.0095


,platform,key,posts,gain
1,tiktok,organic · 1-10,184,0.0000
2,tiktok,organic · 101-1000,281,0.0038
3,tiktok,organic · 11-100,584,0.0025
4,tiktok,organic · >1000,117,0.0043
5,tiktok,promo · 0,50,-0.0061
6,tiktok,promo · 1-10,220,0.0053
7,tiktok,promo · 101-1000,154,0.0173
8,tiktok,promo · 11-100,351,0.0092
9,tiktok,promo · >1000,69,0.0004
11,tiktok,spam · 1-10,88,0.0105


In [6]:
# industries (E69 communities) for reading R85
T69[["community", "brands", "posts", "example brands", "words"]]

,community,brands,posts,example brands,words
0,0,68,3594,"AIG, Ace Hardware, Adidas, Amazon, Amazon Ship...","sprawdź, przez, telefon, yamaha, moto, fridge"
1,1,61,1155,"AB InBev, Ahold Delhaize, Albertsons Companies...","promo, kfc, chipotle, pizza, papa, johns"
2,2,55,1373,"AAA, AMD, Acer, Amazon Prime Video, Apple, Asus","hang, village, animation, wwe, coca, cola"
3,3,39,832,"Aetna, Airbnb, Alamo, Allegiant, American, Ame...","subway, boeing, postal, eagle, usps, airlines"
4,4,39,1493,"Alaska, Aon, Apple TV+, Burger King, Burlingto...","lyft, uber, tel, gig, doordash, beat"
5,5,30,2887,"Accenture, Acronis, Adobe, Anydesk, Autodesk, ...","olympics, boxing, madrid, rugby, flights, chri..."
6,6,25,638,"Acura, Advance Auto Parts, Audi, AutoZone, BMW...","radar, nissan, jeep, rear, chevrolet, chrysler"
7,7,16,215,"AT&T, AT&T Internet, Boost Mobile, Breezeline,...","iphone, mint, xfinity, starlink, air, satellite"
8,8,15,313,"BMO, Bank of America, Barclays, Capital One, C...","bmo, truist, residents, citizens, banking, fargo"


## E86 · Brand popularity on Wikipedia
Monthly English Wikipedia pageviews of the brand's article (fetched by `src/fetch_wikipedia.py`; only brand names were sent).

In [7]:
W = cm.wiki_columns(ROOT, raw, meta)
if W is not None:
    w = pd.read_csv(ROOT / "data" / "processed" / "brand_wikipedia.csv"); mb = w.drop_duplicates("brand")
    print("brands:", len(mb), "matched by name:", int((mb.name_match.astype(str) == "True").sum()), "posts with a matched brand:", float((W.wiki_median >= 0).mean().round(3)))
    V86 = cm.wiki_vs_brand(d, W); display(V86.round(3))
    P86 = an.with_columns(f, meta, W, "+ Wikipedia pageviews"); S86 = H.summarise(P86); display(S86.round(3))
    G86 = pd.DataFrame([{"platform": p, "model": k, **dict(zip(["gain", "CI low", "CI high"], H.gain_ci(P86, p, "final model", k)))} for p in cm.PLATS for k in ["+ Wikipedia pageviews"]]); display(G86.round(3))
else:
    V86 = S86 = G86 = pd.DataFrame()

## Takeaways
See `docs/en/experiments.md` (E84-E86). Numbers: `data/processed/context_more_numbers.json`.

In [8]:
json.dump({"columns_models": S84.to_dict("records"), "columns_gains": G84.to_dict("records"), "promo_subgroups": R85.to_dict("records"), "label_gain_by_type": L85.to_dict("records"),
           "label_gain_by_type_and_reactions": L85b.to_dict("records"), "wiki_vs_brand": V86.to_dict("records"), "wiki_models": S86.to_dict("records"), "wiki_gain": G86.to_dict("records")},
          open(ROOT / "data" / "processed" / "context_more_numbers.json", "w"), indent=1, default=float)